# Finetune model on 80-20 train split

model submitted for shared task

In [1]:
from datasets import Dataset, DatasetDict, load_metric
import numpy as np
from pathlib import Path
from transformers import AutoModelForTokenClassification, AutoTokenizer, DataCollatorForTokenClassification, set_seed, TrainingArguments, Trainer

In [2]:
set_seed(42)
"""
    learning rate: 1e-5
    batch size: 32
    optimizer: AdamW
    scheduler: linear
    epochs: 78
    max seq: 500
"""

label2id = {
    "O": 0,
    "B-DISORDER": 1,
    "I-DISORDER": 2,
    "B-DRUG": 3,
    "I-DRUG": 4,
    "B-FUNCTION": 5,
    "I-FUNCTION": 6,
}

id2label = {
    0:'O',
    1:'B-DISORDER',
    2:'I-DISORDER', 
    3:'B-DRUG',
    4:'I-DRUG',
    5:'B-FUNCTION',
    6:'I-FUNCTION'
}

In [3]:
train_ds = ['ja_twjp_520-540_4', 'ja_twjp_240-260_4', 'ja_twjp_460-480_5', 'ja_twjp_380-400_7',
 'ja_twjp_320-340_10', 'ja_twjp_420-440_19', 'ja_twjp_300-320_1', 'ja_twjp_440-460_12',
 'ja_twjp_100-120_3', 'ja_twjp_100-120_18', 'ja_twjp_260-280_9', 'ja_twjp_240-260_12',
 'ja_twjp_200-220_10', 'ja_twjp_120-140_15', 'ja_twjp_100-120_0', 'ja_twjp_500-520_19',
 'ja_twjp_000-020_17', 'ja_twjp_280-300_9', 'ja_twjp_000-020_18', 'ja_twjp_220-240_0',
 'ja_twjp_160-180_19', 'ja_twjp_180-200_14', 'ja_twjp_180-200_19', 'ja_twjp_460-480_2',
 'ja_twjp_060-080_19', 'ja_twjp_160-180_9', 'ja_twjp_140-160_15', 'ja_twjp_060-080_5',
 'ja_twjp_040-060_13', 'ja_twjp_400-420_9', 'ja_twjp_020-040_7', 'ja_twjp_540-560_4',
 'ja_twjp_140-160_14', 'ja_twjp_180-200_17', 'ja_twjp_120-140_12', 'ja_twjp_340-360_4',
 'ja_twjp_000-020_10', 'ja_twjp_220-240_6', 'ja_twjp_140-160_8', 'ja_twjp_020-040_13',
 'ja_twjp_220-240_4', 'ja_twjp_440-460_8', 'ja_twjp_520-540_7', 'ja_twjp_300-320_5',
 'ja_twjp_220-240_17', 'ja_twjp_400-420_15', 'ja_twjp_260-280_18', 'ja_twjp_160-180_14',
 'ja_twjp_300-320_14', 'ja_twjp_460-480_13', 'ja_twjp_400-420_8', 'ja_twjp_040-060_11',
 'ja_twjp_020-040_17', 'ja_twjp_420-440_10', 'ja_twjp_340-360_15', 'ja_twjp_100-120_17',
 'ja_twjp_200-220_7', 'ja_twjp_520-540_5', 'ja_twjp_280-300_8', 'ja_twjp_100-120_7',
 'ja_twjp_380-400_1', 'ja_twjp_480-500_6', 'ja_twjp_060-080_7', 'ja_twjp_060-080_15',
 'ja_twjp_460-480_11', 'ja_twjp_480-500_8', 'ja_twjp_120-140_18', 'ja_twjp_260-280_14',
 'ja_twjp_120-140_17', 'ja_twjp_400-420_2', 'ja_twjp_320-340_18', 'ja_twjp_420-440_2',
 'ja_twjp_020-040_16', 'ja_twjp_140-160_19', 'ja_twjp_260-280_5', 'ja_twjp_020-040_2',
 'ja_twjp_540-560_2', 'ja_twjp_100-120_16', 'ja_twjp_040-060_19', 'ja_twjp_200-220_11',
 'ja_twjp_460-480_17', 'ja_twjp_240-260_7', 'ja_twjp_200-220_9', 'ja_twjp_040-060_0',
 'ja_twjp_320-340_2', 'ja_twjp_400-420_13', 'ja_twjp_340-360_10', 'ja_twjp_380-400_15',
 'ja_twjp_020-040_6', 'ja_twjp_000-020_8', 'ja_twjp_260-280_3', 'ja_twjp_280-300_19',
 'ja_twjp_240-260_11', 'ja_twjp_380-400_8', 'ja_twjp_060-080_14', 'ja_twjp_340-360_7',
 'ja_twjp_240-260_3', 'ja_twjp_340-360_16', 'ja_twjp_200-220_8', 'ja_twjp_220-240_19',
 'ja_twjp_000-020_3', 'ja_twjp_500-520_14', 'ja_twjp_040-060_10', 'ja_twjp_000-020_9',
 'ja_twjp_240-260_18', 'ja_twjp_140-160_17', 'ja_twjp_460-480_18', 'ja_twjp_100-120_4',
 'ja_twjp_120-140_0', 'ja_twjp_520-540_15', 'ja_twjp_260-280_8', 'ja_twjp_300-320_8',
 'ja_twjp_300-320_11', 'ja_twjp_080-100_18', 'ja_twjp_100-120_12', 'ja_twjp_300-320_16',
 'ja_twjp_540-560_18', 'ja_twjp_280-300_18', 'ja_twjp_240-260_8', 'ja_twjp_140-160_5',
 'ja_twjp_500-520_2', 'ja_twjp_360-380_6', 'ja_twjp_080-100_4', 'ja_twjp_280-300_14',
 'ja_twjp_280-300_11', 'ja_twjp_120-140_10', 'ja_twjp_300-320_10', 'ja_twjp_040-060_17',
 'ja_twjp_100-120_14', 'ja_twjp_320-340_14', 'ja_twjp_160-180_15', 'ja_twjp_480-500_15',
 'ja_twjp_420-440_7', 'ja_twjp_140-160_12', 'ja_twjp_420-440_1', 'ja_twjp_440-460_3',
 'ja_twjp_280-300_15', 'ja_twjp_500-520_18', 'ja_twjp_220-240_3', 'ja_twjp_140-160_0',
 'ja_twjp_420-440_11', 'ja_twjp_540-560_14', 'ja_twjp_040-060_9', 'ja_twjp_240-260_2',
 'ja_twjp_120-140_19', 'ja_twjp_540-560_11', 'ja_twjp_480-500_12', 'ja_twjp_500-520_10',
 'ja_twjp_240-260_16', 'ja_twjp_340-360_2', 'ja_twjp_500-520_0', 'ja_twjp_420-440_6',
 'ja_twjp_280-300_7', 'ja_twjp_080-100_8', 'ja_twjp_380-400_2', 'ja_twjp_260-280_19',
 'ja_twjp_180-200_18', 'ja_twjp_340-360_3', 'ja_twjp_540-560_10', 'ja_twjp_260-280_17',
 'ja_twjp_140-160_18', 'ja_twjp_020-040_19', 'ja_twjp_220-240_14', 'ja_twjp_420-440_14',
 'ja_twjp_380-400_3', 'ja_twjp_260-280_10', 'ja_twjp_120-140_1', 'ja_twjp_220-240_8',
 'ja_twjp_180-200_8', 'ja_twjp_120-140_4', 'ja_twjp_280-300_1', 'ja_twjp_420-440_15',
 'ja_twjp_080-100_17', 'ja_twjp_340-360_14', 'ja_twjp_020-040_1', 'ja_twjp_140-160_16',
 'ja_twjp_420-440_17', 'ja_twjp_320-340_5', 'ja_twjp_020-040_8', 'ja_twjp_100-120_9',
 'ja_twjp_400-420_6', 'ja_twjp_420-440_9', 'ja_twjp_160-180_5', 'ja_twjp_360-380_11',
 'ja_twjp_020-040_3', 'ja_twjp_020-040_18', 'ja_twjp_160-180_0', 'ja_twjp_160-180_7',
 'ja_twjp_060-080_0', 'ja_twjp_140-160_11', 'ja_twjp_200-220_1', 'ja_twjp_020-040_5',
 'ja_twjp_380-400_10', 'ja_twjp_300-320_17', 'ja_twjp_540-560_5', 'ja_twjp_480-500_14',
 'ja_twjp_400-420_4', 'ja_twjp_260-280_15', 'ja_twjp_460-480_10', 'ja_twjp_340-360_17',
 'ja_twjp_280-300_4', 'ja_twjp_080-100_19', 'ja_twjp_420-440_8', 'ja_twjp_220-240_18',
 'ja_twjp_480-500_3', 'ja_twjp_460-480_19', 'ja_twjp_340-360_8', 'ja_twjp_340-360_19',
 'ja_twjp_160-180_10', 'ja_twjp_500-520_6', 'ja_twjp_440-460_0', 'ja_twjp_080-100_1',
 'ja_twjp_340-360_11', 'ja_twjp_120-140_13', 'ja_twjp_360-380_1', 'ja_twjp_360-380_17',
 'ja_twjp_440-460_2', 'ja_twjp_300-320_0', 'ja_twjp_020-040_0', 'ja_twjp_000-020_19',
 'ja_twjp_360-380_0', 'ja_twjp_200-220_14', 'ja_twjp_260-280_1', 'ja_twjp_260-280_7',
 'ja_twjp_080-100_16', 'ja_twjp_160-180_16', 'ja_twjp_120-140_6', 'ja_twjp_060-080_13',
 'ja_twjp_140-160_1', 'ja_twjp_520-540_2', 'ja_twjp_140-160_3', 'ja_twjp_360-380_3',
 'ja_twjp_340-360_9', 'ja_twjp_440-460_13', 'ja_twjp_160-180_2', 'ja_twjp_000-020_6',
 'ja_twjp_280-300_17', 'ja_twjp_400-420_0', 'ja_twjp_180-200_9', 'ja_twjp_000-020_14',
 'ja_twjp_240-260_9', 'ja_twjp_140-160_4', 'ja_twjp_340-360_5', 'ja_twjp_260-280_12',
 'ja_twjp_500-520_7', 'ja_twjp_100-120_8', 'ja_twjp_180-200_2', 'ja_twjp_240-260_14',
 'ja_twjp_280-300_13', 'ja_twjp_440-460_17', 'ja_twjp_300-320_15', 'ja_twjp_460-480_8',
 'ja_twjp_320-340_8', 'ja_twjp_440-460_19', 'ja_twjp_540-560_3', 'ja_twjp_220-240_15',
 'ja_twjp_480-500_1', 'ja_twjp_020-040_14', 'ja_twjp_200-220_17', 'ja_twjp_060-080_4',
 'ja_twjp_320-340_16', 'ja_twjp_420-440_18', 'ja_twjp_440-460_14', 'ja_twjp_400-420_11',
 'ja_twjp_040-060_14', 'ja_twjp_020-040_12', 'ja_twjp_180-200_10', 'ja_twjp_100-120_15',
 'ja_twjp_080-100_14', 'ja_twjp_200-220_3', 'ja_twjp_280-300_0', 'ja_twjp_440-460_9',
 'ja_twjp_360-380_19', 'ja_twjp_220-240_7', 'ja_twjp_500-520_5', 'ja_twjp_320-340_1',
 'ja_twjp_160-180_18', 'ja_twjp_180-200_15', 'ja_twjp_280-300_10', 'ja_twjp_120-140_14',
 'ja_twjp_160-180_3', 'ja_twjp_400-420_3', 'ja_twjp_240-260_15', 'ja_twjp_240-260_1',
 'ja_twjp_440-460_18', 'ja_twjp_320-340_19', 'ja_twjp_020-040_11', 'ja_twjp_460-480_0',
 'ja_twjp_340-360_12', 'ja_twjp_360-380_2', 'ja_twjp_200-220_0', 'ja_twjp_200-220_4',
 'ja_twjp_400-420_7', 'ja_twjp_520-540_14', 'ja_twjp_220-240_5', 'ja_twjp_220-240_12',
 'ja_twjp_300-320_6', 'ja_twjp_460-480_7', 'ja_twjp_100-120_5', 'ja_twjp_380-400_4',
 'ja_twjp_280-300_5', 'ja_twjp_380-400_5', 'ja_twjp_400-420_1', 'ja_twjp_480-500_11',
 'ja_twjp_400-420_17', 'ja_twjp_300-320_2', 'ja_twjp_480-500_7', 'ja_twjp_360-380_9',
 'ja_twjp_280-300_6', 'ja_twjp_160-180_11', 'ja_twjp_380-400_17', 'ja_twjp_160-180_1',
 'ja_twjp_020-040_15']

In [4]:
valid_ds = ['ja_twjp_360-380_18',  'ja_twjp_080-100_10', 'ja_twjp_160-180_17', 'ja_twjp_480-500_18',
 'ja_twjp_240-260_13', 'ja_twjp_300-320_12', 'ja_twjp_360-380_12', 'ja_twjp_520-540_13',
 'ja_twjp_020-040_4', 'ja_twjp_200-220_13', 'ja_twjp_140-160_10', 'ja_twjp_480-500_10',
 'ja_twjp_440-460_4', 'ja_twjp_080-100_11', 'ja_twjp_180-200_12', 'ja_twjp_040-060_16',
 'ja_twjp_080-100_9', 'ja_twjp_520-540_16', 'ja_twjp_000-020_5', 'ja_twjp_200-220_6',
 'ja_twjp_520-540_9', 'ja_twjp_020-040_9', 'ja_twjp_060-080_12', 'ja_twjp_380-400_6',
 'ja_twjp_160-180_6', 'ja_twjp_080-100_5', 'ja_twjp_360-380_4', 'ja_twjp_440-460_6',
 'ja_twjp_160-180_4', 'ja_twjp_320-340_3', 'ja_twjp_500-520_9', 'ja_twjp_320-340_9',
 'ja_twjp_140-160_6', 'ja_twjp_120-140_9', 'ja_twjp_060-080_11', 'ja_twjp_160-180_8',
 'ja_twjp_260-280_13', 'ja_twjp_480-500_0', 'ja_twjp_440-460_5', 'ja_twjp_180-200_7',
 'ja_twjp_500-520_15', 'ja_twjp_380-400_11', 'ja_twjp_260-280_6', 'ja_twjp_500-520_4',
 'ja_twjp_220-240_16', 'ja_twjp_320-340_11', 'ja_twjp_540-560_0', 'ja_twjp_460-480_16',
 'ja_twjp_140-160_9', 'ja_twjp_540-560_15', 'ja_twjp_400-420_19', 'ja_twjp_180-200_16',
 'ja_twjp_100-120_2', 'ja_twjp_340-360_13', 'ja_twjp_440-460_10', 'ja_twjp_340-360_6',
 'ja_twjp_280-300_3', 'ja_twjp_040-060_3', 'ja_twjp_440-460_16', 'ja_twjp_540-560_12',
 'ja_twjp_380-400_18', 'ja_twjp_480-500_17', 'ja_twjp_000-020_1', 'ja_twjp_360-380_15',
 'ja_twjp_520-540_1', 'ja_twjp_220-240_13', 'ja_twjp_000-020_12', 'ja_twjp_060-080_16',
 'ja_twjp_460-480_1', 'ja_twjp_380-400_16', 'ja_twjp_380-400_12', 'ja_twjp_460-480_4',
 'ja_twjp_120-140_3', 'ja_twjp_320-340_4', 'ja_twjp_120-140_16', 'ja_twjp_020-040_10',
 'ja_twjp_280-300_2', 'ja_twjp_040-060_6', 'ja_twjp_240-260_19']

In [5]:
model_checkpoint = "daisaku-s/medtxt_ner_roberta"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)
smmmodel = AutoModelForTokenClassification.from_pretrained(model_checkpoint, num_labels=7,id2label=id2label, label2id=label2id, ignore_mismatched_sizes=True)

Some weights of BertForTokenClassification were not initialized from the model checkpoint at daisaku-s/medtxt_ner_roberta and are newly initialized because the shapes did not match:
- classifier.weight: found shape torch.Size([79, 768]) in the checkpoint and torch.Size([7, 768]) in the model instantiated
- classifier.bias: found shape torch.Size([79]) in the checkpoint and torch.Size([7]) in the model instantiated
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## Data

In [6]:
ds = DatasetDict()

ds['train'] = Dataset.from_dict({"id": train_ds})
ds['validation'] = Dataset.from_dict({"id": valid_ds})

In [7]:
JA_PATH = Path('train/ja_train')
corpus = {}
for f in JA_PATH.iterdir():
    if f.suffix == '.txt':
        with f.open() as text:
            fields = text.readlines()[0].split(':')
            keyid = fields[0]
            offset = len(keyid)+1
            data = ':'.join(fields[1:])
            if f.stem in corpus:
                corpus[f.stem]['text'] = data
                corpus[f.stem]['offset'] = len(keyid) + 1
            else:
                corpus[f.stem] = {'text': data,
                                  'offset': len(keyid) + 1,
                                  'ann': []}
    elif f.suffix == '.ann':
        with f.open() as anno:
            for line in anno:
                if line.startswith("T"):
                    fields = line.strip().split("\t")
                    if f.stem in corpus:
                        corpus[f.stem]['ann'].append((fields[1], fields[2]))
                    else:
                        corpus[f.stem] = {'ann': [(fields[1], fields[2])]}

In [8]:
ner2id = {'DISORDER': 1, 'DRUG': 3, 'FUNCTION': 5}
def tokenize_and_align_labels(examples):
    global corpus
    final_labels = []
    texts = [corpus[textid]['text'] for textid in examples["id"]]
    tokenized_inputs = tokenizer(texts, max_length=512)
    for idx, tid in enumerate(examples["id"]):
        new_labels = [0] * len(tokenized_inputs[idx].word_ids)
        for label, token in corpus[tid]['ann']:
            tag, ts, te = label.split(' ')
            labelid = ner2id[tag]
            ts = int(ts) - corpus[tid]['offset']
            te = int(te) - corpus[tid]['offset']
            start = tokenized_inputs[idx].char_to_token(ts)
            end = tokenized_inputs[idx].char_to_token(te - 1)
            new_labels[start: end + 1] = [labelid + 1] * (end - start + 1)  # more ids based on drug, disorder or function
            new_labels[start] = labelid
        for widx, word_id in enumerate(tokenized_inputs[idx].word_ids):
            if word_id is None:
                new_labels[widx] = -100
        final_labels.append(new_labels)
    tokenized_inputs["labels"] = final_labels
    return tokenized_inputs

In [9]:
tokenized_datasets = ds.map(tokenize_and_align_labels, batched=True)

Map:   0%|          | 0/313 [00:00<?, ? examples/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


Map:   0%|          | 0/79 [00:00<?, ? examples/s]

## Metric

In [10]:
metric = load_metric("seqeval", trust_remote_code=True)

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    # Remove ignored index (special tokens)
    true_predictions = [
        [id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = metric.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"]
    }

/tmp/ipykernel_95993/826350436.py:1: FutureWarning: load_metric is deprecated and will be removed in the next major version of datasets. Use 'evaluate.load' instead, from the new library 🤗 Evaluate: https://huggingface.co/docs/evaluate
  metric = load_metric("seqeval", trust_remote_code=True)


In [11]:
data_collator = DataCollatorForTokenClassification(tokenizer)

## Train

In [12]:
batch_size = 16
args = TrainingArguments(
    "ja_ner",
    evaluation_strategy = "epoch",
    save_strategy = "epoch",
    learning_rate=1e-5,
    per_device_train_batch_size=batch_size,
    per_device_eval_batch_size=batch_size,
    num_train_epochs=50,
    #weight_decay=0.01,
    #fp16=True,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=3,
    push_to_hub=False,
)


trainer = Trainer(
    smmmodel,
    args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["validation"],
    data_collator=data_collator,
    tokenizer=tokenizer,
    compute_metrics=compute_metrics
)

/home/vahbuna/miniforge3/lib/python3.10/site-packages/transformers/training_args.py:1474: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [13]:
trainer.train()

Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,No log,0.466035,0.652542,0.312373,0.422497
2,No log,0.286997,0.696000,0.705882,0.700906
3,No log,0.234518,0.757576,0.760649,0.759109
4,No log,0.217068,0.759843,0.782961,0.771229
5,No log,0.213097,0.786000,0.797160,0.791541
6,No log,0.208058,0.795547,0.797160,0.796353
7,No log,0.212643,0.798000,0.809331,0.803625
8,No log,0.213714,0.810101,0.813387,0.811741
9,No log,0.225772,0.815195,0.805274,0.810204
10,No log,0.225525,0.812500,0.817444,0.814965


/home/vahbuna/miniforge3/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vahbuna/miniforge3/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/home/vahbuna/miniforge3/lib/python3.10/site-packages/seqeval/metrics/v1.py:57: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


TrainOutput(global_step=1000, training_loss=0.06344997668266296, metrics={'train_runtime': 332.4605, 'train_samples_per_second': 47.073, 'train_steps_per_second': 3.008, 'total_flos': 694086594967656.0, 'train_loss': 0.06344997668266296, 'epoch': 50.0})

In [14]:
trainer.save_model('ja_f1_best')

## Eval on dev set

In [3]:
import torch

In [5]:
JA_DEV = Path('dev/ja_dev')
out_dir = Path('ft80_dev')
for f in JA_DEV.iterdir():
    if f.suffix != '.txt':
        continue
    with f.open() as text:
        annotations = []
        fields = text.readlines()[0].split(':')
        keyid = fields[0]
        offset = len(keyid)+1
        data = ':'.join(fields[1:])
        with torch.inference_mode():
            vecs = tokenizer(data,
                             padding=True, 
                             truncation=True,
                             return_tensors="pt", max_length=512).to('cuda')
            ner_logits = trainer.model(input_ids=vecs["input_ids"], attention_mask=vecs["attention_mask"])
            idx = torch.argmax(ner_logits.logits, dim=2).detach().cpu().numpy().tolist()[0]
            tokens = vecs.tokens()[1: -1]
        labels = [id2label[x] for x in idx][1:-1]
        prev_label = None
        prev_tag = ['', '']
        candidate = []
        start = 0
        for token, label in zip(tokens, labels):
            tag = label.split('-')
            if token.startswith('▁'):
                token = token[1:]
                start += 1
            if tag[0] == 'B':
                candidate= [(start, len(token) + start, token)]
            elif tag[0] != 'B' and len(prev_tag) > 1 and len(tag) > 1 and tag[1] == prev_tag[1]:
                candidate.append((start, len(token) + start, token))
            elif candidate:
                anno = ''.join(ctoken for s, e, ctoken in candidate)
                annotations.append((prev_tag[1], candidate[0][0] + offset - 1, candidate[-1][1] + offset - 1, anno))
                candidate = []
            start += len(token)
            prev_tag = tag
        count = 1
        with (out_dir / (f.stem + '.ann')).open('w') as output:
            for ann in annotations:
                output.write(f"T{count}\t{ann[0]} {ann[1]} {ann[2]}\t{ann[3]}\n")
                count += 1

|ne|tp|fp|fn|precision|recall|f1|
|---|---:|---:|---:|---:|---:|---:|
|DISORDER|326|138|166|0.7026|0.6626|0.6820|
|DRUG|290|37|83|0.8869|0.7775|0.8286|
|FUNCTION|24|37|41|0.3934|0.3692|0.3810|
|all|640|212|290|0.7512|0.6882|0.7183|

|||
|---|---:|
|Task2aMicroP| 0.7512|
|Task2aMicroR| 0.6882|
|Task2aMicroF1| 0.7183|
|Task2aMacroP| 0.6610|
|Task2aMacroR| 0.6031|
|Task2aMacroF1| 0.6305|